# Debug Harmony — run attack + see logs

**Not a Submit.** Calls `AttackAlgorithm().run()` so you see:
- prints under cells (`[hybrid]`, farm lines)
- `/kaggle/working/attack_run_summary.txt` (right **Output** panel → refresh)

## Modes
1. `deterministic` — fast schedule + summary (not LB-realistic); no `llama_cpp` needed
2. `gpt_oss` — attach GGUF + GPU; slower, more realistic; needs `llama_cpp` install

## Steps
1. Competition input (+ GGUF if `gpt_oss`)
2. GPU ON. For `gpt_oss`: **Internet ON** for the install cell, then OK to turn OFF
3. Set `AGENT_MODE = 'gpt_oss'` (and `BUDGET_S`) in knobs cell
4. **Run All** → scroll to last cells; refresh Output for summary file


In [ ]:
import sys, os, glob, time
from pathlib import Path

sys.argv = [sys.argv[0]]

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        p = Path('/kaggle/working/debug_heartbeat.txt')
        p.parent.mkdir(parents=True, exist_ok=True)
        with p.open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

log('path probe')
dataset_root = None
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    break
log(f'dataset_root={dataset_root}')
assert dataset_root, 'Attach competition input'


In [ ]:
ATTACK_B64 = (
    "..."
)

from pathlib import Path
import base64, time

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/debug_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

attack_bytes = base64.b64decode(ATTACK_B64)
out = Path('/kaggle/working/attack.py')
out.write_bytes(attack_bytes)
log(f'wrote {out} ({len(attack_bytes)} bytes)')


In [ ]:
import glob, os, time
from pathlib import Path

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/debug_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

# --- knobs ---
AGENT_MODE = 'gpt_oss'         # 'deterministic' | 'gpt_oss'
# Budget must beat search reserve (~180s in attack). Use >= 600 for full hybrid.
BUDGET_S = 600.0
MAX_TOOL_HOPS = 8
N_CTX = 4096
N_GPU_LAYERS = -1
# Debug-only: shrink reserve so more of BUDGET_S is search/farm
DEBUG_SHRINK_RESERVE = True

def pick_gguf(*needles: str):
    hits = []
    for p in glob.glob('/kaggle/input/**/*.gguf', recursive=True):
        low = p.lower()
        if all(n.lower() in low for n in needles):
            hits.append(p)
    if not hits and needles:
        for p in glob.glob('/kaggle/input/**/*.gguf', recursive=True):
            if needles[0].lower() in p.lower():
                hits.append(p)
    if not hits:
        return None
    hits.sort(key=lambda p: (Path(p).stat().st_size, p))
    return hits[0]

GPT_OSS_GGUF = pick_gguf('gpt-oss') or pick_gguf('gpt_oss') or pick_gguf('oss-20')
log(f'AGENT_MODE={AGENT_MODE} BUDGET_S={BUDGET_S}')
log(f'GPT_OSS_GGUF={GPT_OSS_GGUF}')
if AGENT_MODE == 'gpt_oss' and not GPT_OSS_GGUF:
    raise RuntimeError('AGENT_MODE=gpt_oss but no GGUF — attach model or use deterministic')


In [ ]:
# gpt_oss / GGUF needs llama_cpp. Internet ON for this cell (debug only).
import sys, subprocess, time

def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

if AGENT_MODE != 'gpt_oss':
    log('skip llama_cpp install (deterministic mode)')
else:
    try:
        from llama_cpp import Llama  # noqa: F401
        log('llama_cpp already installed')
    except ModuleNotFoundError:
        log('installing llama-cpp-python (CUDA wheels)…')
        cmds = [
            [
                sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python',
                '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121',
            ],
            [
                sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python',
                '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu122',
            ],
            [sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python'],
        ]
        last_err = None
        for cmd in cmds:
            try:
                subprocess.check_call(cmd)
                from llama_cpp import Llama  # noqa: F401
                log('llama_cpp installed OK')
                break
            except Exception as e:
                last_err = e
                log(f'attempt failed: {type(e).__name__}: {e}')
        else:
            raise RuntimeError(
                'Could not install llama_cpp. Turn Internet ON and re-run this cell.'
            ) from last_err
    log('llama_cpp OK')


In [ ]:
import importlib.util
import time
from pathlib import Path

import aicomp_sdk
from aicomp_sdk import AttackRunConfig
from aicomp_sdk.agents import build_agent
from aicomp_sdk.core.env.sandbox import SandboxEnv
from aicomp_sdk.guardrails.optimal import Guardrail as OptimalGuardrail

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/debug_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass


def load_attack():
    path = Path('/kaggle/working/attack.py')
    spec = importlib.util.spec_from_file_location('kaggle_debug_attack', path)
    mod = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    if DEBUG_SHRINK_RESERVE:
        mod.MARGIN_S = 20.0
        mod.MAX_DYNAMIC_RESERVE_S = 60.0
        log('debug: shrunk MARGIN_S/MAX_DYNAMIC_RESERVE_S for visible search')
    return mod


def make_agent():
    if AGENT_MODE == 'deterministic':
        log('building deterministic agent')
        return build_agent('deterministic')
    if AGENT_MODE == 'gpt_oss':
        from aicomp_sdk.agents.gpt_oss_agent import (
            GPTOSSAgent,
            build_gpt_oss_backend_config,
            build_gpt_oss_parser,
        )
        from aicomp_sdk.agents.hf_chat_template.backends.llama_cpp import (
            LlamaCppChatTemplateBackend,
        )
        log(f'loading llama_cpp gpt_oss from {GPT_OSS_GGUF}')
        t0 = time.time()
        cfg = build_gpt_oss_backend_config(model_path=GPT_OSS_GGUF, local_files_only=True)
        backend = LlamaCppChatTemplateBackend.from_model_path(
            model_path=GPT_OSS_GGUF,
            config=cfg,
            n_ctx=N_CTX,
            n_gpu_layers=N_GPU_LAYERS,
            verbose=False,
        )
        agent = GPTOSSAgent(
            backend,
            parser=build_gpt_oss_parser(None, model_id=cfg.model_id or 'openai/gpt-oss-20b'),
        )
        log(f'loaded gpt_oss in {time.time()-t0:.1f}s')
        return agent
    raise ValueError(AGENT_MODE)


mod = load_attack()
summary = Path('/kaggle/working/attack_run_summary.txt')
if hasattr(mod, 'SUMMARY_PATHS'):
    mod.SUMMARY_PATHS = (summary, Path('attack_run_summary.txt'))

fixtures = Path(aicomp_sdk.__file__).resolve().parent / 'fixtures'
env = SandboxEnv(
    seed=1,
    fixtures_dir=fixtures,
    agent=make_agent(),
    guardrail=OptimalGuardrail(),
)

log(f'running AttackAlgorithm budget={BUDGET_S}s mode={AGENT_MODE}')
t0 = time.time()
findings = mod.AttackAlgorithm().run(
    env,
    AttackRunConfig(time_budget_s=BUDGET_S, max_tool_hops=MAX_TOOL_HOPS),
)
elapsed = time.time() - t0
log(f'done wall_s={elapsed:.1f} findings={len(findings) if findings else 0}')


In [ ]:
from pathlib import Path
import time

def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

summary = Path('/kaggle/working/attack_run_summary.txt')
print('\n======== attack_run_summary.txt ========\n', flush=True)
if summary.exists():
    print(summary.read_text(encoding='utf-8'), flush=True)
    log(f'file also at Output → {summary} (hit refresh)')
else:
    log('MISSING summary')

log(f'findings={len(findings) if findings else 0}')
if findings:
    print('example:', findings[0].user_messages[0][:400], flush=True)

print('\nLook for: core_best=  selected=  double_bare trials/fire/exact', flush=True)
print('deterministic: easy schedule check — gpt_oss: trust selected= more', flush=True)
